# Train BERTimbau Large v5 — SPR 2026

**Objetivo:** Treinar modelo próprio e salvar pesos no formato usado pelos notebooks de inferência.

## ⚙️ Configuração no Kaggle

| Setting | Valor |
|---------|-------|
| **Internet** | OFF |
| **Accelerator** | GPU T4 x2 (ou P100) |
| **Add Input** | Competition data: `spr-2026-mammography-report-classification` |
| **Add Input** | Models → `bertimbau-ptbr-complete` (fabianofilho) |

## 📂 Output gerado

```
/kaggle/working/
└── advanced_outputs_kaggle_5/
    ├── all_results.pkl          ← OOF predictions por fold
    ├── solo_bert_artifacts.pkl  ← temperatura + thresholds otimizados
    └── weights/
        └── bertimbau_large__cb_focal/
            ├── fold_0.pt ... fold_4.pt
            ├── tokenizer/
            └── model_config/
```

## 🏆 Configuração (baseada no winner 0.84027)

- BERTimbau Large + Focal Loss (γ=2.0, α=0.25)
- 5-Fold StratifiedKFold
- MAX_LEN=512
- Temperature calibration via scipy.optimize
- Threshold multiplier optimization (grid search)

In [ ]:
import os
import re
import gc
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import minimize_scalar
from tqdm.auto import tqdm

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG — baseada no winner (0.84027)
# ═══════════════════════════════════════════════════════════════════════════════
SEED        = 42
MAX_LEN     = 512
BATCH_SIZE  = 8       # T4: 8 para BERTimbau Large (fp16)
EPOCHS      = 5
LR          = 2e-5
WEIGHT_DECAY= 0.01
WARMUP_RATIO= 0.1
N_FOLDS     = 5
NUM_CLASSES = 7
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
VERSION     = 5       # model-v5
CONFIG_KEY  = 'bertimbau_large__cb_focal'

# Paths
DATA_DIR    = Path('/kaggle/input/spr-2026-mammography-report-classification')
OUTPUT_BASE = Path(f'/kaggle/working/advanced_outputs_kaggle_{VERSION}')
WEIGHTS_DIR = OUTPUT_BASE / 'weights' / CONFIG_KEY
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# BERTimbau Large — disponível via Kaggle Models (fabianofilho/bertimbau-ptbr-complete)
# Tenta caminhos comuns; ajuste se necessário
MODEL_CANDIDATES = [
    '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
    '/kaggle/input/bert-large-portuguese-cased',
    '/kaggle/input/bertimbau-large',
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if Path(p).exists()), None)
assert MODEL_PATH, (
    'BERTimbau Large não encontrado!\n'
    'Adicione: Add Input → Models → bertimbau-ptbr-complete (fabianofilho)'
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = torch.cuda.is_available()

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device:     {DEVICE}')
print(f'FP16:       {USE_FP16}')
print(f'Model path: {MODEL_PATH}')
print(f'Output:     {OUTPUT_BASE}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR DADOS
# ═══════════════════════════════════════════════════════════════════════════════
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(train_df)} amostras')
print(f'Test:  {len(test_df)} amostras')
print(f'\nDistribuição das classes (train):')
print(train_df['target'].value_counts().sort_index())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PRÉ-PROCESSAMENTO (mesmo do winner notebook)
# ═══════════════════════════════════════════════════════════════════════════════
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

train_texts = [build_input_text(t) for t in train_df['report'].tolist()]
train_labels = train_df['target'].tolist()
test_texts  = [build_input_text(t) for t in test_df['report'].tolist()]

print(f'Textos processados: {len(train_texts)} train, {len(test_texts)} test')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET PyTorch
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FOCAL LOSS
# ═══════════════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()

criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
print(f'FocalLoss: α={FOCAL_ALPHA}, γ={FOCAL_GAMMA}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES DE TREINO / INFERÊNCIA
# ═══════════════════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, leave=False):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        optimizer.zero_grad()
        if USE_FP16:
            with torch.cuda.amp.autocast():
                logits = model(**kwargs).logits
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(**kwargs).logits
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        logits = model(**kwargs).logits
        probs  = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
        if 'labels' in batch:
            all_labels.extend(batch['labels'].numpy())
    return np.array(all_probs), np.array(all_labels) if all_labels else None


def compute_f1(probs, labels):
    preds = probs.argmax(axis=1)
    return f1_score(labels, preds, average='macro')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO: TEMPERATURA + THRESHOLD MULTIPLIERS
# ═══════════════════════════════════════════════════════════════════════════════
def temperature_scale(probs, temperature):
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

def optimize_temperature(oof_probs, oof_labels):
    """Busca temperatura ótima via scipy minimize_scalar."""
    def neg_f1(temp):
        cal  = temperature_scale(oof_probs, temp)
        pred = cal.argmax(axis=1)
        return -f1_score(oof_labels, pred, average='macro')

    result = minimize_scalar(neg_f1, bounds=(0.1, 5.0), method='bounded')
    return result.x

def optimize_thresholds(cal_probs, oof_labels, n_steps=40):
    """Grid search de threshold multipliers por classe."""
    best_thresholds = [1.0] * NUM_CLASSES
    best_f1         = f1_score(oof_labels, cal_probs.argmax(axis=1), average='macro')

    grid = np.linspace(0.05, 3.0, n_steps)
    # Otimizar cada classe independentemente (greedy)
    for c in range(NUM_CLASSES):
        best_t_c = best_thresholds[c]
        for t in grid:
            thresh_try    = best_thresholds.copy()
            thresh_try[c] = t
            preds = apply_thresholds(cal_probs, thresh_try)
            f1    = f1_score(oof_labels, preds, average='macro')
            if f1 > best_f1:
                best_f1    = f1
                best_t_c   = t
        best_thresholds[c] = best_t_c

    return best_thresholds, best_f1

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5-FOLD CV TRAINING
# ═══════════════════════════════════════════════════════════════════════════════
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Salvar tokenizer no output (necessário para inferência)
tok_save_dir = WEIGHTS_DIR / 'tokenizer'
tokenizer.save_pretrained(tok_save_dir)
print(f'Tokenizer salvo em: {tok_save_dir}')

train_arr = np.array(train_texts)
label_arr = np.array(train_labels)

test_ds     = MammographyDataset(test_texts, tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
test_probs = np.zeros((len(test_df), NUM_CLASSES))
all_results  = {}  # OOF por fold
oof_probs_all  = np.zeros((len(train_df), NUM_CLASSES))

for fold, (train_idx, val_idx) in enumerate(skf.split(train_arr, label_arr)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*60}')

    X_tr, y_tr = train_arr[train_idx].tolist(), label_arr[train_idx].tolist()
    X_vl, y_vl = train_arr[val_idx].tolist(),   label_arr[val_idx].tolist()

    train_ds = MammographyDataset(X_tr, y_tr, tokenizer, MAX_LEN)
    val_ds   = MammographyDataset(X_vl, y_vl, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

    # Carregar modelo fresco por fold
    config = AutoConfig.from_pretrained(MODEL_PATH, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, config=config
    ).to(DEVICE)

    # Salvar config do modelo (uma vez)
    if fold == 0:
        config_save_dir = WEIGHTS_DIR / 'model_config'
        config.save_pretrained(config_save_dir)
        print(f'Config salvo em: {config_save_dir}')

    total_steps = len(train_loader) * EPOCHS
    optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )
    scaler = torch.cuda.amp.GradScaler() if USE_FP16 else None

    best_f1    = 0.0
    best_state = None

    for epoch in range(EPOCHS):
        loss = train_epoch(model, train_loader, optimizer, scheduler, scaler)
        val_probs, val_lbls = evaluate(model, val_loader)
        val_f1 = compute_f1(val_probs, val_lbls)
        print(f'  Epoch {epoch+1}/{EPOCHS} | loss={loss:.4f} | val_f1={val_f1:.5f}')

        if val_f1 > best_f1:
            best_f1    = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'  Melhor OOF F1 fold {fold}: {best_f1:.5f}')

    # Salvar pesos do melhor epoch
    weight_path = WEIGHTS_DIR / f'fold_{fold}.pt'
    torch.save(best_state, weight_path)
    print(f'  Pesos salvos: {weight_path}')

    # OOF predictions com o melhor modelo
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    val_probs_best, val_lbls_best = evaluate(model, val_loader)
    oof_probs_all[val_idx] = val_probs_best

    # Acumular probabilidades de teste
    fold_test_probs, _ = evaluate(model, test_loader)
    test_probs += fold_test_probs

    # Guardar resultados por fold
    all_results[f'fold_{fold}'] = {
        'val_probs':  val_probs_best.tolist(),
        'val_labels': val_lbls_best.tolist(),
        'f1_macro':   float(best_f1),
        'val_idx':    val_idx.tolist(),
    }

    del model, best_state, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

test_probs /= N_FOLDS

oof_f1 = compute_f1(oof_probs_all, label_arr)
print(f'\n✅ OOF F1-macro (5 folds): {oof_f1:.5f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRAÇÃO: TEMPERATURA + THRESHOLDS (sobre OOF)
# ═══════════════════════════════════════════════════════════════════════════════
print('Otimizando temperatura...')
best_temp = optimize_temperature(oof_probs_all, label_arr)
print(f'  Temperatura ótima: {best_temp:.4f}')

oof_calibrated = temperature_scale(oof_probs_all, best_temp)
f1_after_temp  = compute_f1(oof_calibrated, label_arr)
print(f'  F1 após temperatura: {f1_after_temp:.5f}')

print('Otimizando threshold multipliers (greedy por classe)...')
best_thresholds, f1_after_thresh = optimize_thresholds(oof_calibrated, label_arr)
print(f'  Thresholds: {[round(t, 4) for t in best_thresholds]}')
print(f'  F1 após thresholds: {f1_after_thresh:.5f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SALVAR ARTIFACTS (formato compatível com notebooks de inferência)
# ═══════════════════════════════════════════════════════════════════════════════
solo_artifacts = {
    'temperature':  best_temp,
    'thresholds':   best_thresholds,
    'oof_f1_macro': oof_f1,
}

with open(OUTPUT_BASE / 'solo_bert_artifacts.pkl', 'wb') as f:
    pickle.dump(solo_artifacts, f)

with open(OUTPUT_BASE / 'all_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)

print('Artifacts salvos:')
print(f'  solo_bert_artifacts.pkl → temp={best_temp:.4f}, thresholds={[round(t,4) for t in best_thresholds]}')
print(f'  all_results.pkl        → OOF predictions de {N_FOLDS} folds')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GERAR SUBMISSION
# ═══════════════════════════════════════════════════════════════════════════════
test_calibrated = temperature_scale(test_probs, best_temp)
predictions     = apply_thresholds(test_calibrated, best_thresholds)

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'\nsubmission.csv salvo ({len(submission)} linhas)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFICAR ESTRUTURA DO OUTPUT
# ═══════════════════════════════════════════════════════════════════════════════
print(f'Estrutura do output ({OUTPUT_BASE}):')
total = 0
for root, dirs, files in os.walk(OUTPUT_BASE):
    dirs.sort()
    level  = Path(root).relative_to(OUTPUT_BASE).parts
    indent = '  ' * len(level)
    print(f'{indent}{Path(root).name}/')
    for fname in sorted(files):
        fpath = Path(root) / fname
        size  = fpath.stat().st_size
        total += size
        print(f'{indent}  {fname}  ({size/1024/1024:.1f} MB)')

print(f'\nTotal: {total/1024/1024:.0f} MB')
print(f'\n✅ Treino concluído!')
print(f'   OOF F1-macro: {oof_f1:.5f}')
print(f'   Temperatura:  {best_temp:.4f}')
print(f'   Thresholds:   {[round(t,4) for t in best_thresholds]}')
print(f'\n➡️  Próximos passos:')
print(f'   1. Save Version → Save & Run All (Commit)')
print(f'   2. Aba Output → New Dataset → nome: model-v{VERSION}')
print(f'   3. Nos notebooks de resubmit, trocar WEIGHTS_BASE para:')
print(f"      Path('/kaggle/input/model-v{VERSION}/advanced_outputs_kaggle_{VERSION}')")